# FastViT → ADE20K semantic segmentation

Fine-tunes an ImageNet-pretrained FastViT backbone + Semantic FPN decoder on ADE20K (150 classes),
following Sec. 4.4 / Table 9 of the FastViT paper. Runs on the local GPU.

Use the `.venv` kernel (Python environment in the repo root).
Training runs as a detached background process, so it survives closing the notebook or restarting the kernel.
Re-run the training cell after an interruption and it resumes from `last.pth`.

## 1. Environment and GPU

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Run everything from the repo root, wherever the notebook was opened from.
ROOT = Path.cwd()
while not (ROOT / 'segmentation' / 'train_seg.py').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
PY = sys.executable

import torch
print('repo:', ROOT)
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv

## 2. Settings

| Model | Paper mIoU (Table 9) |
|---|---|
| fastvit_sa12 | 38.0 |
| fastvit_sa24 | 41.0 |
| fastvit_sa36 | 42.9 |
| fastvit_ma36 | 44.6 |

The paper schedule is 40k iterations at batch 32 (`BATCH_SIZE * ACCUM_STEPS`).
If you shorten `ITERS`, report it in your write-up, because mIoU will be lower than in the paper.

`CHECKPOINT` must be an **unfused** ImageNet checkpoint, not a `*_reparam.pth.tar` file.
Set it to `None` to download the one for `MODEL` automatically.

In [ ]:
MODEL = 'fastvit_sa12'
ITERS = 40000
BATCH_SIZE = 8      # lower to 4 if you get CUDA out-of-memory (then set ACCUM_STEPS = 8)
ACCUM_STEPS = 4     # effective batch = BATCH_SIZE * ACCUM_STEPS = 32, as in the paper
WORKERS = 8
CHECKPOINT = f'checkpoints/{MODEL}.pth.tar'
OUTPUT = f'segmentation/runs/{MODEL}'

if CHECKPOINT and not os.path.exists(CHECKPOINT):
    print(CHECKPOINT, 'not found, downloading the unfused ImageNet weights instead')
    CHECKPOINT = None
os.makedirs(OUTPUT, exist_ok=True)
ckpt_arg = f'--checkpoint "{CHECKPOINT}"' if CHECKPOINT else ''
resume_arg = f'--resume "{OUTPUT}/last.pth"' if os.path.exists(f'{OUTPUT}/last.pth') else ''
print('resuming' if resume_arg else 'starting fresh', '->', OUTPUT)

## 3. (Optional) Sanity check

Runs 20 iterations on 64 images to confirm everything works before the long run. **Only needed once.**
The first run also downloads ADE20K (~1 GB) into the Hugging Face cache.

In [ ]:
!{PY} segmentation/train_seg.py --model {MODEL} {ckpt_arg} --iters 20 --batch-size 2 --accum-steps 1 \
    --train-samples 64 --eval-interval 20 --eval-samples 10 --final-eval-samples 10 --log-interval 5 \
    --workers {WORKERS} --amp --output segmentation/runs/debug_run

## 4. Train

Starts training in the background and returns immediately. Output goes to `OUTPUT/train.log`.
Saves `last.pth` every 500 iterations and `best.pth` whenever validation mIoU improves
(on 500 val images during training; the full 2,000 at the end).
To resume after an interruption, re-run cells 1–2, then this cell.

In [ ]:
cmd = (f'{PY} segmentation/train_seg.py --model {MODEL} {ckpt_arg} {resume_arg} '
       f'--iters {ITERS} --batch-size {BATCH_SIZE} --accum-steps {ACCUM_STEPS} --amp '
       f'--eval-interval 2000 --eval-samples 500 --save-interval 500 --workers {WORKERS} '
       f'--output "{OUTPUT}"')
pid_file = Path(OUTPUT) / 'train.pid'
running = pid_file.exists() and Path(f'/proc/{pid_file.read_text().strip()}').exists()
if running:
    print('already running, pid', pid_file.read_text().strip())
else:
    log = open(f'{OUTPUT}/train.log', 'a')
    proc = subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    pid_file.write_text(str(proc.pid))
    print('started pid', proc.pid, '\n', cmd)

Check progress (re-run any time). To stop training: `!kill $(cat {OUTPUT}/train.pid)`.

In [ ]:
!tail -n 15 {OUTPUT}/train.log

## 5. Training curves

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv(f'{OUTPUT}/log.csv')
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
t = log.dropna(subset=['loss']); ax[0].plot(t['iter'], t['loss']); ax[0].set(title='train loss', xlabel='iteration')
e = log.dropna(subset=['mIoU']); ax[1].plot(e['iter'], e['mIoU'], 'o-'); ax[1].set(title='val mIoU (%)', xlabel='iteration')
plt.tight_layout(); plt.savefig(f'{OUTPUT}/curves.png', dpi=150); plt.show()

## 6. Evaluate on the full val set + structural reparameterization

Evaluates the trained (multi-branch) model and the reparameterized (fused) model.
mIoU should be identical, while the fused model is faster. That's the paper's central idea.

In [ ]:
!{PY} segmentation/eval_seg.py --checkpoint "{OUTPUT}/best.pth" --reparam --benchmark --vis 8

## 7. Look at predictions (image | ground truth | prediction)

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f'{OUTPUT}/vis/*.png'))[:8]:
    display(Image(p, width=900))

In [ ]:
import json
r = json.load(open(f'{OUTPUT}/eval_results.json'))
print({k: round(r['unfused'][k], 2) for k in ('mIoU', 'aAcc', 'mAcc')}, 'fused:', {k: round(r['fused'][k], 2) for k in ('mIoU',)})
print('latency (ms):', r['latency_ms'])